# **Lecture:** Tracking and video analysis

![](multi-object-tracking.jpg)

### **1. Learning Objectives**


By the end of this lab, you should be able to:

- Explain the difference between **detection** and **tracking**.
- Describe the role of **classical trackers** (KCF, CSRT) and when they are useful.
- Implement a simple **tracking-by-detection** pipeline using PyTorch and OpenCV.
- Run experiments on a **video** and observe how tracking behaves over time.
- Understand how tracking and detection relate to **real systems** like the DeepStream-based pedestrian analytics system described in the master’s thesis proposal (age and gender classification over detected people).


### **2. Detection vs Tracking**

#### 2.1 Object Detection

**Object detection** answers the question: *“What objects are in this frame, and where are they?”*.

- Input: a single image (frame).
- Output: bounding boxes + class labels (e.g., `person`).
- Typical models: Faster R-CNN, YOLO, SSD, etc.
- Cost: relatively **expensive** per frame (deep network forward pass).

#### 2.2 Object Tracking

**Object tracking** answers the question: *“Where did this object move between frames?”*.

- Input: an initial bounding box in frame *t*.
- Output: updated bounding box in frame *t+1, t+2, ...*.
- Trackers assume the object is the **same** and try to follow it over time.
- Cost: usually **cheaper** than running a full detector every frame.

Tracking is useful when:
- You want **smooth trajectories** of objects.
- You want to **reduce computation** by not running detection on every frame.
- You want to maintain **identity** (ID) of each person over time.

#### 2.3 Classical Trackers: KCF and CSRT

OpenCV provides several classical trackers, including:

- **KCF (Kernelized Correlation Filter)**
  - Fast, works well when the object appearance does not change too much.
  - Sensitive to occlusions and large scale changes.

- **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
  - More accurate than KCF in many cases.
  - Slower than KCF, but still real-time on CPU for moderate resolutions.

These trackers are **not deep learning models**; they are classical computer vision algorithms.

#### 2.4 Tracking-by-Detection

**Tracking-by-detection** combines both ideas:

1. Run a **detector** every N frames (e.g., every 5 or 10 frames).
2. Initialize or update **trackers** for each detected object.
3. Between detection frames, use the trackers to **propagate** object positions.

Advantages:
- More robust than pure tracking (because detection can recover from drift).
- More efficient than running detection on every frame.
- Natural fit for systems that need **IDs over time**, like counting people entering a store or tracking their paths.

In the master’s thesis system, a similar idea is used: detect people, then classify age and gender for each detection, and aggregate statistics over time in a real deployment.

In [4]:
video_path = '/home/mateo/Documentos/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4'  # Replace with your video path

In [2]:
from IPython.display import HTML
HTML("""
<video width="640" height="480" controls>
    <source src="/home/mateo/Documentos/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4">
</video>
""")

### **KCF (Kernelized Correlation Filter)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Track one object based only on its appearance from the first frame

In [1]:
%pip uninstall opencv-python -y
%pip install opencv-contrib-python

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Note: you may need to restart the kernel to use updated packages.
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 MB 4.8 MB/s  0:00:16m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
from ultralytics import YOLO

# Cargar modelo
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

# Inicializar tracker
tracker = cv2.TrackerKCF.create()
tracker.init(frame, bbox)

# LOOP DE VISUALIZACIÓN
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, "Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("KCF Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 111.6ms
Speed: 3.5ms preprocess, 111.6ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)


QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for ex

### **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Improved version of KCF that adapts to scale and appearance changes

In [4]:
import cv2
from ultralytics import YOLO

# Cargar modelo YOLO
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:  # clase "car"
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box (xyxy → xywh)
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

print("BBox inicial:", bbox)

# 🔥 Inicializar CSRT (más robusto que KCF)
tracker = cv2.TrackerCSRT_create()

# (si falla, usa esta alternativa)
# tracker = cv2.legacy.TrackerCSRT_create()

tracker.init(frame, bbox)

# LOOP de tracking
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,0,0), 2)
        cv2.putText(frame, "CSRT Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("CSRT Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 206.3ms
Speed: 6.6ms preprocess, 206.3ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)
BBox inicial: (1309, 412, 145, 101)


### **YOLO + ByteTrack**
- Type: **Multi-Object Tracking (MOT)**
- Requires: **Detection on every frame (YOLO)**
- Uses IoU + confidence score for association

**Key idea:** Fast and efficient tracking-by-detection without deep appearance features

In [ ]:
from ultralytics import YOLO

# Load an official or custom model
model = YOLO("yolo26n.pt")  # Load an official Detect model

#results = model.track(source="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4", show=True)  # Tracking with default tracker - BoT-SORT
results = model.track("race_car.mp4", show=True, tracker="bytetrack.yaml")  # with ByteTrack


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 37.8ms
video 1/1 (frame 2/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 25.2ms
video 1/1 (frame 3/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 23.8ms
video 1/1 (frame 4/210) /Users/eugenio/Documents/Compute

: 

### **Summary**

| Method | Type | Uses Detection Every Frame? | # Objects | Robustness |
|-------|------|----------------------------|----------|-----------|
| KCF | SOT | ❌ No | 1 | Low |
| CSRT | SOT | ❌ No | 1 | Medium |
| YOLO + ByteTrack | MOT | ✅ Yes | Many | High |

**One-Line Intuition**

- KCF → "Follow this object fast"
- CSRT → "Follow this object carefully"
- YOLO + ByteTrack → "Track many objects efficiently"

### **The Tracking Challenge: Motion Trail + Heatmap + Smart Counter**

- Work in **groups of 2 students**

#### Objective

Build a **multi-feature tracking system** using a video of people (the video will be provided by the instructor) that includes:

1. Motion Trail (trajectory visualization)
2. Heatmap of activity
3. Smart Entry Counter

#### Task Overview

You will use an object tracking method (e.g., YOLO + ByteTrack) to:

- Track people across frames
- Extract their positions over time
- Build three different visual/analytical outputs

#### **Task 1: Motion Trail (Trajectory)**

Requirements:
- For each tracked person:
  - Compute the **center point** of the bounding box
  - Draw a **trail (path)** of previous positions
- The trail must:
  - Fade over time (older points disappear gradually)
  - Be updated frame-by-frame

#### **Task 2: Heatmap of Activity**

Requirements:
- Accumulate all detected center points into a spatial grid
- Generate a **heatmap visualization** showing:
  - High-density areas (more traffic)
  - Low-density areas
- Overlay the heatmap on the video or display separately

### **Task 3: Smart Entry Counter**

- Define a **virtual line** (e.g., entrance)
- Count how many people:
  - Cross the line in a specific direction
- Avoid double counting:
  - Each tracked ID should be counted only once

### **Deliverables**

Each group must submit:

1. Working code
2. Video output
3. Short explanation (ONLY ONE of the following per student) of the working logic:
   - Motion Trail
   - Heatmap
   - Counter logic

### **Grading Rubric (10 points total)**

#### Functionality (6 points)

| Component | Points | Criteria |
|----------|--------|---------|
| Motion Trail | 2 pts | Works correctly (center + fading) OR not |
| Heatmap | 2 pts | Correct accumulation and visualization OR not |
| Counter | 2 pts | Correct counting (no duplicates) OR not |

#### Explanation (4 points)

| Criteria | Points |
|--------|--------|
| Clear and correct explanation of chosen component | 4 pts |
| Incorrect or unclear explanation | 0 pts |
- No partial credit

In [1]:
import cv2
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# 1. Cargar el modelo YOLO
# Nota: En tu notebook decía "yolo26n.pt", asegúrate de usar el archivo correcto de tu PC
model = YOLO("yolo26n.pt") 

# 2. Diccionario para guardar el historial de puntos de cada ID
# defaultdict evita errores si el ID es nuevo y no tiene historial previo
track_history = defaultdict(lambda: [])

# Longitud máxima de la cola (para el efecto de desvanecimiento)
max_trail = 50 

cap = cv2.VideoCapture("store.mp4")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # 3. Ejecutar YOLO con ByteTrack
    # persist=True es vital: le dice al tracker que recuerde los IDs del frame anterior
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)

    # Verificar si se detectó algo con ID
    if results[0].boxes.id is not None:
        # Extraer cajas y convertirlas a enteros
        boxes = results[0].boxes.xyxy.cpu().numpy()
        # Extraer los IDs de seguimiento
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            
            # 4. Calcular el punto central de la caja
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2.5)

            # Agregar el punto al historial de este ID
            track_history[track_id].append((center_x, center_y))
            
            # Mantener solo los últimos 'max_trail' puntos (fading espacial)
            if len(track_history[track_id]) > max_trail:
                track_history[track_id].pop(0)

            # --- DIBUJO ---
            # Dibujar la caja y el ID
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            class_ids = results[0].boxes.cls.int().cpu().tolist()
            class_id = class_ids[track_ids.index(track_id)]
            cv2.putText(frame, f"ID: {track_id} | Class: {class_id}", (int(x1), int(y1)-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Dibujar el rastro (Motion Trail)
# Dibujar el rastro (Motion Trail) con desvanecimiento de color
            points = track_history[track_id]
            for i in range(1, len(points)):
                # 1. Grosor fijo para toda la línea
                thickness = 3 
                
                # 2. Calcular qué tan reciente es el punto (ratio de 0.0 a 1.0)
                # i=1 es el punto más viejo de la cola, i=len(points)-1 es el más nuevo
                ratio = i / len(points)
                
                # 3. Calcular la intensidad del color (de 0 a 255)
                # OpenCV usa BGR, así que modificamos el último valor (Rojo)
                intensidad_rojo = int(255 * ratio)
                color_fade = (0, 0, intensidad_rojo)
                
                cv2.line(frame, points[i - 1], points[i], color_fade, thickness)

    cv2.imshow("Task 1: Motion Trail", frame)

    # Presiona ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for ex

In [8]:
import cv2
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

model = YOLO("yolo26n.pt") 
cap = cv2.VideoCapture("store.mp4")

# --- NUEVO: 1. Preparar el Lienzo del Mapa de Calor ---
# Obtenemos las dimensiones exactas del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# Matriz de ceros en formato decimal (float) para poder sumar sin límite
accum_heatmap = np.zeros((height, width), dtype=np.float32)

track_history = defaultdict(lambda: [])
max_trail = 30 

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)

    # Creamos una capa vacía solo para el calor de los objetos en ESTE frame
    temp_heatmap = np.zeros((height, width), dtype=np.float32)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            # --- TASK 1: Motion Trail (Desvanecimiento) ---
            track_history[track_id].append((center_x, center_y))
            if len(track_history[track_id]) > max_trail:
                track_history[track_id].pop(0)

            points = track_history[track_id]
            for i in range(1, len(points)):
                ratio = i / len(points)
                intensidad_rojo = int(255 * ratio)
                cv2.line(frame, points[i - 1], points[i], (0, 0, intensidad_rojo), 3)

            # --- NUEVO: 2. El "Fuego" (Añadir calor al Task 2) ---
            # Dibujamos un círculo sólido invisible de valor 1 en el centro de la persona
            cv2.circle(temp_heatmap, (center_x, center_y), radius=25, color=1, thickness=-1)

    # Sumamos el calor de este frame al lienzo acumulado de todo el video
    accum_heatmap += temp_heatmap

    # --- NUEVO: CREAR LA VISUALIZACIÓN DEL MAPA DE CALOR ---
    # a. Suavizar los bordes (Blur) para que se vea como manchas de calor reales
    heatmap_blurred = cv2.GaussianBlur(accum_heatmap, (51, 51), 0)

    # b. El Termómetro (Normalizar de 0 a 255)
    max_calor = np.max(heatmap_blurred)
    if max_calor > 0:
        heatmap_norm = (heatmap_blurred / max_calor * 255).astype(np.uint8)
    else:
        heatmap_norm = heatmap_blurred.astype(np.uint8)

    # c. Aplicar la paleta de colores de calor (Azul=frío, Rojo=caliente)
    heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)

    # d. Truco Pro: Evitar que todo el video se vea azul oscuro
    # Creamos una máscara para aislar solo las zonas que tienen "algo" de calor
    mask = heatmap_norm > 5 
    
    # Clonamos el frame para no dañar el original
    final_frame = frame.copy()
    
    # Mezclamos el video original (50%) con los colores del mapa (50%) 
    # SOLO en las zonas donde la máscara nos dice que hay actividad
    final_frame[mask] = cv2.addWeighted(frame, 0.5, heatmap_color, 0.5, 0)[mask]

    # Mostrar resultado
    cv2.imshow("Tracking: Trail & Heatmap", final_frame)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

model = YOLO("yolo26n.pt") 
cap = cv2.VideoCapture("store.mp4")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
accum_heatmap = np.zeros((height, width), dtype=np.float32)

track_history = defaultdict(lambda: [])
max_trail = 50 

# --- TASK 3: Variables del Contador ---
# Dibujaremos la línea un poco más abajo de la mitad de la pantalla
# --- TASK 3: Variables del Contador Bidireccional ---
line_y = int(height * 0.6) 
entradas = 0
salidas = 0
ids_entraron = set()
ids_salieron = set()

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # CORRECCIÓN DE FALSOS POSITIVOS: Solo personas (classes=[0]) y alta confianza (conf=0.5)
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=[0], conf=0.5, device=0, verbose=False)
    temp_heatmap = np.zeros((height, width), dtype=np.float32)

    # --- DIBUJAR LA LÍNEA DEL CONTADOR Y EL TEXTO ---
    cv2.line(frame, (0, line_y), (width, line_y), (0, 255, 255), 2)
    cv2.putText(frame, f"Entradas: {entradas} | Salidas: {salidas}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            # --- Task 1: Motion Trail ---
            track_history[track_id].append((center_x, center_y))
            if len(track_history[track_id]) > max_trail:
                track_history[track_id].pop(0)

            points = track_history[track_id]
            for i in range(1, len(points)):
                ratio = i / len(points)
                cv2.line(frame, points[i - 1], points[i], (0, 0, int(255 * ratio)), 3)

            # --- Task 2: Heatmap Fuego ---
            cv2.circle(temp_heatmap, (center_x, center_y), radius=25, color=1, thickness=-1)

            # --- TASK 3: Smart Counter (Entradas y Salidas) ---
            if len(points) >= 2:
                prev_y = points[-2][1]
                curr_y = points[-1][1]

                # CASO A: ENTRADA (Cruza de arriba hacia abajo)
                if prev_y < line_y and curr_y >= line_y:
                    if track_id not in ids_entraron:
                        entradas += 1
                        ids_entraron.add(track_id)

                # CASO B: SALIDA (Cruza de abajo hacia arriba)
                elif prev_y > line_y and curr_y <= line_y:
                    if track_id not in ids_salieron:
                        salidas += 1
                        ids_salieron.add(track_id)

    # Procesar y dibujar Heatmap
    accum_heatmap += temp_heatmap
    heatmap_blurred = cv2.GaussianBlur(accum_heatmap, (51, 51), 0)
    
    max_calor = np.max(heatmap_blurred)
    if max_calor > 0:
        heatmap_norm = (heatmap_blurred / max_calor * 255).astype(np.uint8)
    else:
        heatmap_norm = heatmap_blurred.astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
    mask = heatmap_norm > 5 
    
    final_frame = frame.copy()
    final_frame[mask] = cv2.addWeighted(frame, 0.5, heatmap_color, 0.5, 0)[mask]

    cv2.imshow("Tracking: Trail, Heatmap & Counter", final_frame)

    # CORRECCIÓN DE VELOCIDAD: Reducimos a 1ms la espera para liberar los FPS
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>